# 02 — EDA chuyên sâu (Home Credit)

**Mục tiêu:** hiểu phân phối, tương quan và cơ chế thiếu dữ liệu trước khi chọn kỹ thuật xử lý.

**Rubric:** Bảng — EDA thống kê, kiểm định phân phối, MCAR (10%).

**Đề yêu cầu:**
- [ ] Kiểm định phân phối cho thuộc tính số: Shapiro–Wilk nếu n ≤ 5000, D’Agostino–Pearson nếu n > 5000; phân loại chuẩn/không chuẩn theo p-value.
- [ ] Heatmap Pearson và Spearman; xác định cặp đa cộng tuyến mạnh (|r| > 0.9) và đề xuất xử lý.
- [ ] Trực quan giá trị thiếu bằng `missingno`; Little’s MCAR test; phân loại MCAR/MAR/MNAR kèm giải thích.

**Đầu vào:** development set từ split của notebook 01; `HomeCredit_columns_description.csv`.

**Đầu ra:** `results/tabular/tables/normality_tests.csv`, `high_correlation_pairs.csv`, hình heatmap/missingno; danh sách quyết định cho các notebook sau.
---
*Quy ước khung:* ô **[Code]** sẽ được thay bằng ô code; ô **[Nhận xét]** chỉ viết sau khi có kết quả thật. Mọi score/p-value phải được **in ra và gán vào biến Python** (yêu cầu của đề).

## 0. Thiết lập

> **[Code]** Import thư viện; tìm gốc repo (`find_project_root`, nên chuyển vào `src/data_utils.py`); đọc `data/raw/home_credit/application_train.csv`; nạp `data/splits/home_credit_split_ids.csv` và **chỉ lấy phần development**. Không dùng phần test trong notebook này. `SEED = 42`.

## 1. Giá trị đặc biệt cần làm rõ trước khi kiểm định

**Lý thuyết — cần trình bày:**
- Đọc mô tả cột: các cột `DAYS_*` âm vì tính **số ngày trước ngày nộp hồ sơ**.
- `DAYS_EMPLOYED = 365243` (55.374 hồ sơ trong toàn bộ dữ liệu, gần như toàn bộ là hưu trí) là mã quy ước, không phải số ngày thật.
- `CODE_GENDER = "XNA"` (4 hồ sơ), các giá trị `XNA` khác nếu có.
- Phân biệt *lỗi dữ liệu* với *mã quy ước* với *giá trị cực đoan thật*.

> **[Code]** Đếm số hồ sơ có từng giá trị đặc biệt trên development; tạo bản phân tích trong đó mã quy ước được thay bằng NaN và thêm cờ chỉ báo (ví dụ `DAYS_EMPLOYED_ANOM`). Không sửa file gốc.

> **[Nhận xét]** Mỗi giá trị đặc biệt là gì, có liên quan tới nhãn không, và sẽ xử lý thế nào ở các notebook sau?

## 2. Phân phối thuộc tính số

**Lý thuyết — cần trình bày:**
- Phân biệt biến liên tục, biến đếm, biến nhị phân (`FLAG_*`) — biến nhị phân không cần kiểm định chuẩn.
- Skewness và kurtosis; vì sao biến tiền tệ thường lệch phải.

> **[Code]** Histogram + boxplot cho các biến số chính (`AMT_*`, `DAYS_*`, `EXT_SOURCE_*`, `OWN_CAR_AGE`…); bảng skewness/kurtosis. Lưu hình vào `results/tabular/figures/`.

> **[Nhận xét]** Biến nào lệch mạnh? Có cần biến đổi log/quantile không? Liên hệ với lựa chọn scaler ở notebook 05.

## 3. Kiểm định phân phối chuẩn

**Lý thuyết — cần trình bày:**
- Giả thuyết $H_0$: mẫu đến từ phân phối chuẩn; mức ý nghĩa $\alpha = 0.05$.
- Shapiro–Wilk: $W = \frac{(\sum a_i x_{(i)})^2}{\sum (x_i - \bar x)^2}$, dùng khi $n \le 5000$.
- D’Agostino–Pearson: $K^2 = Z^2(\text{skew}) + Z^2(\text{kurt}) \sim \chi^2_2$, dùng khi $n > 5000$.
- Lưu ý: với n rất lớn, độ lệch rất nhỏ cũng cho p-value nhỏ → kết hợp Q-Q plot và skewness khi diễn giải.

> **[Code]** Duyệt các cột số liên tục; chọn kiểm định theo số giá trị không thiếu n; lưu bảng (cột, n, kiểm định, thống kê, p-value, kết luận) vào biến `normality_results` và file CSV; vẽ Q-Q plot cho vài cột tiêu biểu.

> **[Nhận xét]** Bao nhiêu cột được xem là chuẩn? Hệ quả cho việc dùng Pearson hay Spearman, Z-score hay IQR, và chọn scaler.

## 4. Tương quan và đa cộng tuyến

**Lý thuyết — cần trình bày:**
- Pearson: $r = \frac{\text{cov}(X,Y)}{\sigma_X \sigma_Y}$ — đo quan hệ tuyến tính, nhạy với ngoại lai.
- Spearman: Pearson trên thứ hạng — đo quan hệ đơn điệu.
- Đa cộng tuyến và tác động lên mô hình tuyến tính.

> **[Code]** Tính ma trận Pearson và Spearman, vẽ heatmap (có thể lọc nhóm cột để dễ đọc); liệt kê các cặp |r| > 0.9 vào biến `high_corr_pairs` và file CSV.

> **[Nhận xét]** Các cặp tương quan mạnh có ý nghĩa gì (ví dụ nhóm `*_AVG`/`*_MODE`/`*_MEDI`, `AMT_CREDIT`–`AMT_GOODS_PRICE`)? Đề xuất xử lý: giữ một đại diện, bỏ bớt, hay PCA — thực hiện ở notebook 06.

## 5. Giá trị thiếu: trực quan và cơ chế thiếu

**Lý thuyết — cần trình bày:**
- Định nghĩa MCAR, MAR, MNAR kèm ví dụ trong dữ liệu này.
- Little’s MCAR test: thống kê $d^2 \sim \chi^2$; $H_0$: dữ liệu thiếu hoàn toàn ngẫu nhiên. Giả định gần chuẩn đa biến.
- Giới hạn: test chỉ bác bỏ hoặc không bác bỏ MCAR, **không phân biệt được MAR và MNAR**.

> **[Code]** `missingno` (matrix, bar, heatmap tương quan thiếu); chạy Little’s test trên nhóm cột chọn lọc (có thể phải tự cài đặt theo công thức; nếu lấy mẫu thì ghi rõ số dòng và seed); so sánh tỷ lệ `TARGET=1` giữa nhóm thiếu và không thiếu; kiểm tra `OWN_CAR_AGE` theo `FLAG_OWN_CAR`.

> **[Nhận xét]** Phân loại từng nhóm cột (nhà ở `*_AVG`…, `EXT_SOURCE_*`, `OWN_CAR_AGE`, `OCCUPATION_TYPE`) vào MCAR/MAR/MNAR có lập luận. Thiếu `OWN_CAR_AGE` là "không áp dụng" — ảnh hưởng thế nào tới cách điền?

## Tổng kết và lựa chọn

> **[Nhận xét]** Liệt kê các quyết định chuyển sang notebook sau: xử lý giá trị đặc biệt, cột cần bỏ/gộp, nhóm cột cần cách điền riêng.

## Hạn chế

> **[Nhận xét]** Kiểm định trên n lớn, giả định của Little’s test, phần phân tích chỉ chạy trên mẫu (nếu có).